# Muon frente a AdamW

Notebook opcional del **módulo 21** del curso (en la app: `uv run dev` y la ruta `#/modulo/optimizacion`). Usa modelos reales; el módulo interactivo explica la teoría con NumPy.

> **Requisitos:** desde la raíz del repositorio, `uv sync --group notebooks` y abre este notebook con `uv run --group notebooks jupyter lab`. La primera ejecución descarga los modelos de Hugging Face.

In [ ]:
import matplotlib.pyplot as plt
import torch
import torch.nn as nn
import torch.nn.functional as F


def newton_schulz(G, pasos=5):
    a, b, c = 3.4445, -4.7750, 2.0315
    X = G / (G.norm() + 1e-7)
    t = X.shape[0] > X.shape[1]
    if t:
        X = X.T
    for _ in range(pasos):
        A = X @ X.T
        X = a * X + (b * A + c * A @ A) @ X
    return X.T if t else X


class Muon(torch.optim.Optimizer):
    def __init__(self, params, lr=0.02, momentum=0.95):
        super().__init__(params, dict(lr=lr, momentum=momentum))

    @torch.no_grad()
    def step(self):
        for g in self.param_groups:
            for p in g["params"]:
                if p.grad is None:
                    continue
                m = self.state[p].setdefault("m", torch.zeros_like(p))
                m.mul_(g["momentum"]).add_(p.grad)
                escala = max(1, p.shape[0] / p.shape[1]) ** 0.5
                p.add_(newton_schulz(m), alpha=-g["lr"] * escala)

In [ ]:
torch.manual_seed(0)
X = torch.randn(4096, 32)
y = (X[:, :4].sum(1) > 0).long()


def entrenar(usar_muon):
    torch.manual_seed(1)
    red = nn.Sequential(nn.Linear(32, 256), nn.ReLU(), nn.Linear(256, 256), nn.ReLU(), nn.Linear(256, 2))
    matrices = [p for p in red.parameters() if p.ndim == 2][1:-1]     # capas ocultas -> Muon
    resto = [p for p in red.parameters() if all(p is not q for q in matrices)]
    opts = [Muon(matrices), torch.optim.AdamW(resto, 1e-3)] if usar_muon else [torch.optim.AdamW(red.parameters(), 1e-3)]
    curva = []
    for paso in range(300):
        i = torch.randint(0, len(X), (128,))
        perdida = F.cross_entropy(red(X[i]), y[i])
        for o in opts:
            o.zero_grad()
        perdida.backward()
        for o in opts:
            o.step()
        curva.append(perdida.item())
    return curva


plt.plot(entrenar(False), label="AdamW")
plt.plot(entrenar(True), label="Muon (+AdamW)")
plt.legend()
plt.yscale("log")
plt.show()

**ART + RULER (esqueleto).** Con la librería `openpipe-art`: se define un `rollout(model, escenario)` que ejecuta el agente y devuelve una `Trajectory`; se generan grupos de trayectorias por escenario, `ruler_score_group(grupo, "openai/o4-mini")` las puntúa de forma relativa y `model.train(grupos)` aplica GRPO. Consulta la documentación de ART para la versión actual de la API.